[← 01 - What Is a Database](<01-what-is-a-database.ipynb>) · [Course Overview](<00-course-overview.ipynb>)

# 02 - SQL Basics: SELECT, WHERE, and ORDER BY

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samietex/SQL_Tutorial/blob/main/02-sql-basics.ipynb)

Here's a question a small shop might ask: which completed orders were over $100, biggest first? Answering it takes three pieces of SQL, `SELECT`, `WHERE`, and `ORDER BY`, and this notebook builds the query one piece at a time on a table small enough to check by hand. `01 - What Is a Database` ended by asking you to look at any table and say what one row represents. Here, one row is one order, and every query below is a way of choosing which of those orders you want to see.

*New here? The [Course Overview](<00-course-overview.ipynb>) explains how to run these notebooks. Viewing this on GitHub shows a preview with saved results; it won't execute anything.*

> **By the end of this notebook you'll be able to:** choose columns with `SELECT`, keep rows with `WHERE` and combine conditions with `AND`, `OR`, and `IN`, sort with `ORDER BY`, explain why a row with a missing value falls on neither side of a filter, and recognize the quiet mistakes that return a wrong answer without an error.

> **Read the full lesson:** this notebook is the hands-on companion to [Lesson 2 of the All About Data & More curriculum](https://aaboutdata.substack.com/p/sql-lesson-2-the-question-behind), which walks through the reasoning in more depth.

## 0. Set up

Run the cell below once each time you open the notebook. After it, any cell that starts with `%%sql` runs SQLite against a small lesson database that lives in memory, so there's nothing to install, and any cell that starts with `%%tsql` runs against SQL Server and AdventureWorks, which only section 10 uses and which has its own one-step setup there.

In [1]:
# Run this cell first. It downloads the small helper that lets the cells below run SQL,
# fetching a fresh copy every time so you always have the latest version.
import os, sys, tempfile, urllib.request
helper_dir = os.path.join(tempfile.gettempdir(), 'sql_tutorial_helper')
os.makedirs(helper_dir, exist_ok=True)
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/samietex/SQL_Tutorial/assets/sql_tutorial.py',
    os.path.join(helper_dir, 'sql_tutorial.py'),
)
sys.path.insert(0, helper_dir)
%reload_ext sql_tutorial

Ready. %%sql runs SQLite 3.50.4 in memory, %%tsql runs SQL Server.


## 1. Seven orders to work with

This is an invented shop, not data from a real business. Each row is one order, and the totals are whole dollars to keep the arithmetic easy. These are the same seven rows the newsletter lesson uses, so every result below should match it exactly. Run the setup once; for a clean start later, run `%sql_reset` in a new cell and then run this cell again.

You don't need to understand this cell yet, since creating tables and adding rows get their own notebook in `08a`. It builds the seven orders as a temporary table:

In [2]:
%%sql
CREATE TEMP TABLE lesson_orders (
    order_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    status TEXT NOT NULL,
    total INTEGER
);
INSERT INTO lesson_orders VALUES
    (1, 'Maria', 'completed', 120),
    (2, 'Maria', 'completed', 100),
    (3, 'Maria', 'cancelled', 90),
    (4, 'David', 'completed', 150),
    (5, 'David', 'completed', NULL),
    (6, 'Amina', 'completed', 200),
    (7, 'Amina', 'cancelled', 50);

7 rows inserted


| order_id | customer_name | status | total |
| --- | --- | --- | --- |
| 1 | Maria | completed | 120 |
| 2 | Maria | completed | 100 |
| 3 | Maria | cancelled | 90 |
| 4 | David | completed | 150 |
| 5 | David | completed | NULL |
| 6 | Amina | completed | 200 |
| 7 | Amina | cancelled | 50 |

`NULL` in order 5 means the total is missing. It doesn't mean zero; it means nobody recorded a number, and that one row is going to matter later in this notebook.

## 2. SELECT: choosing the columns

The smallest useful query names some columns and the table they come from:

In [3]:
%%sql
SELECT
    customer_name,
    total
FROM lesson_orders;

customer_name,total
Maria,120
Maria,100
Maria,90
David,150
David,NULL
Amina,200
Amina,50


`SELECT` lists the columns you want, `FROM` names the table, and the semicolon ends the statement. This returns all seven orders, but only two columns of each, so it hasn't removed a single row; it has only decided what you look at.

You'll also see `SELECT *`, where `*` means every column:

In [4]:
%%sql
SELECT *
FROM lesson_orders;

order_id,customer_name,status,total
1,Maria,completed,120
2,Maria,completed,100
3,Maria,cancelled,90
4,David,completed,150
5,David,completed,NULL
6,Amina,completed,200
7,Amina,cancelled,50


It's a fine way to take a first look at a table you don't know. In a query someone else will read, or one you'll come back to in a month, naming the columns is better, because it says which ones the answer depends on, and the result doesn't quietly change shape if somebody adds a column to the table later.

## 3. WHERE: choosing the rows

`SELECT` picks columns, and `WHERE` picks rows:

In [5]:
%%sql
SELECT
    order_id,
    customer_name,
    total
FROM lesson_orders
WHERE total > 100;

order_id,customer_name,total
1,Maria,120
4,David,150
6,Amina,200


The database checks the condition `total > 100` against each row in turn and keeps a row only if the condition is true for it. Maria's order 2 is exactly 100, and 100 isn't greater than 100, so it's left out; if you wanted it, you'd write `>=`. The other comparisons are `<`, `<=`, `=`, and `<>` for "not equal".

Text values go in single quotes, so filtering on the status looks like `WHERE status = 'completed'`, and two conditions can be combined with `AND`, which keeps a row only when both are true:

In [6]:
%%sql
SELECT
    order_id,
    customer_name,
    total
FROM lesson_orders
WHERE status = 'completed' AND total > 100;

order_id,customer_name,total
1,Maria,120
4,David,150
6,Amina,200


On this table the result is the same three rows, because both cancelled orders, 90 and 50, were already under 100. The `AND` still earns its place, though, since it says what you meant: a cancelled order of $300 next month would slip into the first query and stay out of this one.

## 4. Edge case: the order that's neither over nor under $100

You've seen `WHERE total > 100` return three orders. Now run the opposite:

In [7]:
%%sql
SELECT
    order_id,
    customer_name,
    total
FROM lesson_orders
WHERE total <= 100;

order_id,customer_name,total
2,Maria,100
3,Maria,90
7,Amina,50


Another three: orders 2, 3, and 7. Three and three is six, and the table has seven rows.

The missing one is David's order 5. Its total is `NULL`, and the [SQLite expression documentation](https://www.sqlite.org/lang_expr.html) says comparison operators "generally evaluate to NULL when any operand is NULL." So for order 5, `total > 100` isn't true and it isn't false; it's **unknown**, and `WHERE` keeps only rows where the condition is true. The opposite condition is unknown for it too, so order 5 drops out of both.

<p align="center">
  <img src="graphics/02_where_three_outcomes.png" width="620" alt="WHERE total > 100 sorts the seven orders into true (orders 1, 4, 6, kept), false (orders 2, 3, 7, dropped), and unknown (order 5, dropped by both filters)">
</p>

That leads to a common trap. If you go looking for the missing row with `= NULL`, you get no rows at all, for the same reason: comparing anything to `NULL` with `=` gives unknown, never true.

In [8]:
%%sql
SELECT
    order_id,
    customer_name
FROM lesson_orders
WHERE total = NULL;

order_id,customer_name


No rows, and no error to warn you. The way to ask "is this missing?" is a separate operator:

In [9]:
%%sql
SELECT
    order_id,
    customer_name
FROM lesson_orders
WHERE total IS NULL;

order_id,customer_name
5,David


That returns order 5, David. `IS NULL` and `IS NOT NULL` are the only comparisons that give a plain yes or no for a missing value, which is why the same documentation says an `IS` expression can never evaluate to `NULL` itself.

## 5. ORDER BY: sorting what's left

The [SQLite SELECT documentation](https://www.sqlite.org/lang_select.html) says that without an `ORDER BY`, "the order in which the rows are returned is undefined," so if the order of the result matters, you have to ask for it:

In [10]:
%%sql
SELECT
    order_id,
    customer_name,
    total
FROM lesson_orders
WHERE status = 'completed'
ORDER BY
    total DESC;

order_id,customer_name,total
6,Amina,200
4,David,150
1,Maria,120
2,Maria,100
5,David,NULL


`DESC` sorts from largest to smallest, and `ASC`, or leaving the direction off, sorts from smallest to largest. Order 5 is back, since `status = 'completed'` is true for it, and it sits at the bottom. The same documentation explains why: "SQLite considers NULL values to be smaller than any other values for sorting purposes," so missing values come first in an ascending sort and last in a descending one.

Not every engine agrees. The [PostgreSQL documentation](https://www.postgresql.org/docs/current/sql-select.html) says its default is "to act as though nulls are larger than non-nulls," so the same query on PostgreSQL would put order 5 at the top. If the position of missing values matters to whoever reads the result, say so in the query:

In [11]:
%%sql
SELECT
    order_id,
    customer_name,
    total
FROM lesson_orders
WHERE status = 'completed'
ORDER BY
    total ASC NULLS LAST;

order_id,customer_name,total
2,Maria,100
1,Maria,120
4,David,150
6,Amina,200
5,David,NULL


Sorted smallest first, and order 5 still at the bottom, because `NULLS LAST` overrides where missing values go. SQLite has supported it since version 3.30.0 and PostgreSQL supports it too, but SQL Server 2022 doesn't, which section 10 comes back to.

## 6. The order you write it isn't the order it runs

You write a query as `SELECT`, then `FROM`, then `WHERE`, then `ORDER BY`. The database works through it in a different order: it starts `FROM` the table, keeps the rows that pass `WHERE`, builds the columns in `SELECT`, and sorts with `ORDER BY` last. Most of the time you don't need to think about this, and then one day it explains an error.

<p align="center">
  <img src="graphics/02_written_vs_run_order.png" width="480" alt="You write SELECT, FROM, WHERE, ORDER BY; the database runs FROM, then WHERE, then SELECT, where the alias is created, then ORDER BY, which can use it">
</p>

A **column alias** is a name you give a result column with `AS`. Because `ORDER BY` runs after `SELECT`, it can use the alias:

In [12]:
%%sql
SELECT
    order_id,
    total * 0.9 AS discounted_total
FROM lesson_orders
WHERE status = 'completed'
ORDER BY
    discounted_total DESC;

order_id,discounted_total
6,180.0
4,135.0
1,108.0
2,90.0
5,NULL


That sorts the completed orders by their total after a 10 percent discount: order 6 at 180.0 first, order 5's `NULL` last. `WHERE`, though, runs before `SELECT`, so in standard SQL the alias doesn't exist yet when the rows are being filtered. PostgreSQL's documentation puts the rule plainly: an output column's name can be used in `ORDER BY` "but not in the WHERE or HAVING clauses; there you must write out the expression instead." SQLite is more lenient:

In [13]:
%%sql
SELECT
    order_id,
    total * 0.9 AS discounted_total
FROM lesson_orders
WHERE discounted_total > 100;

order_id,discounted_total
1,108.0
4,135.0
6,180.0


It ran, and that's exactly the kind of query that works on your laptop and fails on the company database; section 10 runs it on SQL Server to prove it. Writing the expression itself in `WHERE` works everywhere:

In [14]:
%%sql
SELECT
    order_id,
    total * 0.9 AS discounted_total
FROM lesson_orders
WHERE total * 0.9 > 100;

order_id,discounted_total
1,108.0
4,135.0
6,180.0


## 7. Common mistake: mixing AND and OR without parentheses

Suppose you want Maria's and Amina's orders over $100, and you write it the way you'd say it:

In [15]:
%%sql
SELECT
    order_id,
    customer_name,
    total
FROM lesson_orders
WHERE customer_name = 'Maria' OR customer_name = 'Amina' AND total > 100;

order_id,customer_name,total
1,Maria,120
2,Maria,100
3,Maria,90
6,Amina,200


Maria's 100 and her cancelled 90 are in there, and neither is over $100. `AND` binds more tightly than `OR` (it sits above `OR` in the precedence list in SQLite's expression documentation), so the database read the condition as "Maria, or (Amina and over 100)." Every Maria order passes on the first half alone.

<p align="center">
  <img src="graphics/02_and_or_grouping.png" width="440" alt="Without parentheses SQL groups Amina AND total > 100 first and ORs it with Maria, returning four rows; with parentheses around Maria OR Amina it returns two">
</p>

Parentheses say what you meant:

In [16]:
%%sql
SELECT
    order_id,
    customer_name,
    total
FROM lesson_orders
WHERE (customer_name = 'Maria' OR customer_name = 'Amina') AND total > 100;

order_id,customer_name,total
1,Maria,120
6,Amina,200


That returns orders 1 and 6, Maria's 120 and Amina's 200. A habit worth keeping is to add parentheses any time a `WHERE` mixes `AND` and `OR`, even when you're sure of the precedence, because the next reader might not be.

When the `OR`s are all checking one column against a list of values, `IN` says the same thing more compactly and sidesteps the trap entirely, since the list is one condition:

In [17]:
%%sql
SELECT
    order_id,
    customer_name,
    total
FROM lesson_orders
WHERE customer_name IN ('Maria', 'Amina') AND total > 100;

order_id,customer_name,total
1,Maria,120
6,Amina,200


Same two rows. `NOT IN` keeps the rows whose value isn't in the list, with one catch worth remembering for later: if the list itself contains a `NULL`, `NOT IN` returns nothing at all, for the same unknown-is-not-true reason as section 4.

## 8. Common mistake: a value that's almost right

In [18]:
%%sql
SELECT
    order_id,
    customer_name,
    total
FROM lesson_orders
WHERE status = 'Completed';

order_id,customer_name,total


Zero rows, with a capital C. There's no error, just an empty result, which looks exactly like "there are no completed orders." SQLite compares text using its `BINARY` collation unless told otherwise, and the [datatype documentation](https://www.sqlite.org/datatype3.html) describes it as comparing the data with `memcmp()`, byte for byte, so `'Completed'` and `'completed'` are different values. Other engines can be set up to compare text without regard to case (section 10 shows SQL Server doing exactly that), so this behaves differently depending on where you run it.

The fix is a habit: check the actual values before you filter on them, rather than typing what you expect them to be. `SELECT DISTINCT` returns each different value once:

In [19]:
%%sql
SELECT DISTINCT status
FROM lesson_orders;

status
completed
cancelled


Two values, both lowercase, and now you know what to type.

## 9. Try it yourself

Write each query in a new cell (click **+ Code** in the toolbar, then start the cell with `%%sql`) before looking at the [solutions notebook](<solutions/02-sql-basics-solutions.ipynb>). Predict the result first, then run it; the gap between your prediction and the actual result is where the learning happens.

1. **Predict the rows.** Without running it, say which orders this returns, then run it to check:
   ```sql
   SELECT
       order_id,
       customer_name,
       total
   FROM lesson_orders
   WHERE status = 'completed' AND total < 150;
   ```
2. **David's and Amina's big orders.** Return the completed orders from David or Amina with a total of at least 150, biggest first. You should get two rows.
3. **Orders that need a look.** A manager wants every order that is either not completed or has no recorded total. Return them sorted by `order_id`. (Think about which operator finds a missing total before you start.)
4. **Stretch, on AdventureWorks.** `Production.Product` has 504 products and a `Color` column. Count the black products (`WHERE Color = 'Black'`) and the products that aren't black (`WHERE Color <> 'Black'`). Do the two counts add up to 504? If not, write the query that finds the rest. Save this one for after section 10, since it needs the AdventureWorks setup cell there. (To count rows, use `SELECT COUNT(*) FROM ...`; `03 - Aggregation` explains it properly.)

## 10. Scale it up: the same traps in AdventureWorks

Seven orders make every rule checkable by hand, and real data is where the same rules decide whether your numbers are right. This section runs the lesson's traps against **AdventureWorks2022**, Microsoft's sample database for an invented bike company, on SQL Server. Three things change from here on: tables live in schemas (`Sales.SalesOrderHeader`, `Person.Person`), the dialect is T-SQL, which writes `SELECT TOP 5` where SQLite writes `LIMIT 5` at the end, and the results are far too big to eyeball, which is the whole point.

### Set up AdventureWorks

SQL Server is a separate program from this notebook, so it needs one setup step before the `%%tsql` cells can run:

- **In Google Colab**, the cell below installs SQL Server and AdventureWorks inside your Colab session for you. The first run takes a few minutes and prints each step as it goes. Colab gives you a fresh machine every session, so run it again whenever you come back to the notebook later.
- **On your own machine**, it checks that SQL Server is reachable and tells you what's missing if it isn't; the [Course Overview](<00-course-overview.ipynb>) covers installing SQL Server and AdventureWorks locally.

If you'd rather not set anything up, every `%%tsql` cell below keeps its saved result, so you can read the section without running it.

In [20]:
# Run this once before any %%tsql cell.
# In Google Colab it installs SQL Server and AdventureWorks into this session (about 3 to 5
# minutes the first time, and again in each new session, because Colab starts fresh).
# On your own machine it just checks that your SQL Server is reachable.
%setup_adventureworks

AdventureWorks2022 is ready, so every %%tsql cell in this notebook will run.


### The missing rows, at scale

Some orders were sold by a salesperson, and each has that person's ID in `SalesPersonID`. How many orders did salesperson 279 sell, and how many did everybody else sell?

In [21]:
%%tsql
SELECT COUNT(*) AS orders_by_279
FROM Sales.SalesOrderHeader
WHERE SalesPersonID = 279;

orders_by_279
429


In [22]:
%%tsql
SELECT COUNT(*) AS orders_by_everyone_else
FROM Sales.SalesOrderHeader
WHERE SalesPersonID <> 279;

orders_by_everyone_else
3377


In [23]:
%%tsql
SELECT COUNT(*) AS all_orders
FROM Sales.SalesOrderHeader;

all_orders
31465


429 plus 3,377 is 3,806, out of 31,465 orders. It's David's order 5 again, but this time 27,659 orders are missing from both sides, nearly nine orders in ten:

In [24]:
%%tsql
SELECT COUNT(*) AS orders_with_no_salesperson
FROM Sales.SalesOrderHeader
WHERE SalesPersonID IS NULL;

orders_with_no_salesperson
27659


These are online orders, which no salesperson handled. A report built on `SalesPersonID <> 279` that claims to show "everyone else's sales" leaves out most of the business, and nothing about the query looks wrong.

### The alias that works on your laptop

Section 6 ran an alias in `WHERE` on SQLite and it worked. Here's the same idea on SQL Server, a cell that's meant to fail:

In [25]:
%%tsql --expect-error
SELECT
    SalesOrderID,
    TotalDue * 0.9 AS discounted_total
FROM Sales.SalesOrderHeader
WHERE discounted_total > 100000;

Error (this cell is meant to fail): Invalid column name 'discounted_total'.


"Invalid column name": SQL Server follows the standard rule, because the alias doesn't exist yet when `WHERE` runs. Writing out the expression works on both engines:

In [26]:
%%tsql
SELECT
    SalesOrderID,
    TotalDue * 0.9 AS discounted_total
FROM Sales.SalesOrderHeader
WHERE TotalDue * 0.9 > 150000
ORDER BY
    discounted_total DESC;

SalesOrderID,discounted_total
51131,168739.04250
55282,163816.76448
46616,153461.40201


### Text that compares without regard to case

On SQLite, `'Completed'` didn't match `'completed'`. Here are two searches for a surname on SQL Server, one in lowercase:

In [27]:
%%tsql
SELECT COUNT(*) AS lowercase_smith
FROM Person.Person
WHERE LastName = 'smith';

lowercase_smith
103


In [28]:
%%tsql
SELECT COUNT(*) AS capitalized_smith
FROM Person.Person
WHERE LastName = 'Smith';

capitalized_smith
103


103 both times. AdventureWorks uses the collation `SQL_Latin1_General_CP1_CI_AS`, where `CI` means case-insensitive, so SQL Server treats the two spellings as equal; another database on the same server could be set up the other way. The same query can be right on one engine and silently empty on another, which is why checking the actual values, as in section 8, is the habit that holds up everywhere.

### And the sort

SQL Server, like SQLite, treats `NULL` as smaller than any value when sorting, so the orders with no salesperson come first in an ascending sort:

In [29]:
%%tsql
SELECT TOP 5
    SalesOrderID,
    SalesPersonID,
    TotalDue
FROM Sales.SalesOrderHeader
ORDER BY
    SalesPersonID,
    SalesOrderID;

SalesOrderID,SalesPersonID,TotalDue
43697,NULL,3953.9884
43698,NULL,3756.9890
43699,NULL,3756.9890
43700,NULL,772.5036
43701,NULL,3756.9890


SQL Server 2022 doesn't accept `NULLS LAST`, so when the position of missing values matters there, the usual workaround is to sort on an extra expression first. `CASE WHEN SalesPersonID IS NULL THEN 1 ELSE 0 END` turns each row into a 0 (has a salesperson) or a 1 (doesn't), and sorting on that before the real column pushes the missing values to the end:

In [30]:
%%tsql
SELECT TOP 5
    SalesOrderID,
    SalesPersonID,
    TotalDue
FROM Sales.SalesOrderHeader
ORDER BY
    CASE WHEN SalesPersonID IS NULL THEN 1 ELSE 0 END,
    SalesPersonID,
    SalesOrderID;

SalesOrderID,SalesPersonID,TotalDue
43849,274,23130.2957
44082,274,2297.0332
44508,274,4723.1073
44532,274,2417.4793
45317,274,68918.2404


## Before you trust a filter

- Read your `WHERE` back as a sentence and check it's the question you were asked.
- Ask what happens to rows where the column is `NULL`; they fail every comparison except `IS NULL`.
- Put parentheses around any `OR` that sits next to an `AND`, or use `IN` for a list of values.
- Look at the actual values with `SELECT DISTINCT` before you filter on text.
- If the order of the result matters, write `ORDER BY`; without it, the order is undefined.

## Go deeper

- **[SQL expressions](https://www.sqlite.org/lang_expr.html)** and **[SELECT](https://www.sqlite.org/lang_select.html)**, SQLite. The references for everything the `%%sql` cells do, including the operator precedence list and how `NULL` behaves in comparisons.
- **[Three-valued logic](https://modern-sql.com/concept/three-valued-logic)**, Markus Winand. A clear explanation of the true, false, and unknown idea behind section 4, with the cases where it surprises experienced developers too.
- **[SELECT](https://learn.microsoft.com/en-us/sql/t-sql/queries/select-transact-sql)**, **[WHERE](https://learn.microsoft.com/en-us/sql/t-sql/queries/where-transact-sql)**, and **[Operator precedence](https://learn.microsoft.com/en-us/sql/t-sql/language-elements/operator-precedence-transact-sql)**, Microsoft Learn. The SQL Server references behind section 10, including the logical processing order of a query.
- **[PostgreSQL SELECT](https://www.postgresql.org/docs/current/sql-select.html)**. Where the alias rule and the "nulls are larger" sorting default come from.
- **Practice:** [SQLBolt](https://sqlbolt.com/lesson/select_queries_introduction) lessons 1 to 4, [SQLZoo: SELECT basics](https://sqlzoo.net/wiki/SELECT_basics), and [PostgreSQL Exercises: Basic](https://pgexercises.com/questions/basic/) all check your answers in the browser.

For interview questions on this topic with model answers, see the [study guide](<study-guides/02-sql-basics.md>).

## What's next

You can now choose columns, keep rows, and sort what's left, and you know where `NULL` hides. `03 - Aggregation` keeps the same seven orders and asks a question none of this notebook's clauses can answer on their own: which customers spent more than $200? That needs the rows for each customer combined into one answer, and that's what `GROUP BY` does.

---

[← 01 - What Is a Database](<01-what-is-a-database.ipynb>) · [Course Overview](<00-course-overview.ipynb>) · [03 - Aggregation →](<03-aggregation.ipynb>)

*SQL_Tutorial* is written and maintained by Samuel Shaibu as part of *All About Data & More*. Licensed under [MIT](LICENSE).